In [4]:
import plotly.express as px
import plotly.io as pio
import pandas as pd

from mainnet_launch.database.schema.full import (
    DestinationStates,
    Destinations,
    IncentiveTokenSwapped,
    IncentiveTokenBalanceUpdated,
    IncentiveTokenPrices,
    Tokens,
    Blocks,
)
from mainnet_launch.database.postgres_operations import (
    get_full_table_as_df_with_block,
    get_full_table_as_df,
    get_full_table_as_df_with_tx_hash,
)
from mainnet_launch.constants import (
    BASE_CHAIN,
    ARBITRUM_CHAIN,
    ETH_CHAIN,
    USDC,
    WETH,
    ChainData,
    LIQUIDATION_ROW,
    LIQUIDATION_ROW2,
    ALL_CHAINS,
    PLASMA_CHAIN,
    DEAD_ADDRESS,
    USDC,
)

from mainnet_launch.constants import ChainData
from mainnet_launch.abis import TOKEMAK_LIQUIDATION_ROW_ABI
from mainnet_launch.database.schema.ensure_tables_are_current.using_onchain.not_order_dependent.about_incentives.update_destination_vault_balance_updated import (
    fetch_new_balance_updated_events,
)

from mainnet_launch.data_fetching.alchemy.get_events import fetch_events
from mainnet_launch.database.schema.ensure_tables_are_current.using_onchain.helpers.update_blocks import (
    ensure_all_blocks_are_in_table,
)


pio.templates.default = "plotly_white"


def fetch_liquidation_data(chain: ChainData):
    """
    Fetch liquidation and balance update data for a given chain.

    Parameters:
    -----------
    chain : ChainData
        The chain to fetch data for

    Returns:
    --------
    tuple: (liquidated_df, balance_updated_df, blocks_df)
    """

    balance_updated = fetch_new_balance_updated_events(
        chain.block_autopool_first_deployed, chain.get_block_near_top(), chain
    )

    lr = chain.client.eth.contract(address=LIQUIDATION_ROW(chain), abi=TOKEMAK_LIQUIDATION_ROW_ABI)
    liquidated = fetch_events(lr.events.VaultLiquidated, chain)

    lr2 = chain.client.eth.contract(address=LIQUIDATION_ROW2(chain), abi=TOKEMAK_LIQUIDATION_ROW_ABI)
    liquidated2 = fetch_events(lr2.events.VaultLiquidated, chain)

    # Merge both liquidation dataframes
    liquidated = pd.concat([liquidated, liquidated2], ignore_index=True)

    ensure_all_blocks_are_in_table(liquidated["block"].to_list(), chain)
    ensure_all_blocks_are_in_table(balance_updated["block"].to_list(), chain)

    blocks = get_full_table_as_df(Blocks, where_clause=Blocks.chain_id == chain.chain_id)
    destinations = get_full_table_as_df(Destinations, where_clause=Destinations.chain_id == chain.chain_id)

    liquidated["datetime"] = liquidated["block"].map(blocks.set_index("block")["datetime"].to_dict())
    balance_updated["datetime"] = balance_updated["block"].map(blocks.set_index("block")["datetime"].to_dict())

    liquidated["vault_name"] = liquidated["vault"].map(
        destinations.set_index("destination_vault_address")["underlying_name"].to_dict()
    )
    liquidated["date"] = liquidated["datetime"].dt.date

    return liquidated, balance_updated


def fetch_and_process_liquidation_by_week(chain: ChainData):
    """
    Fetch liquidation data and aggregate by week for a specific token.

    Parameters:
    -----------
    chain : ChainData
        The chain to fetch data for
    token_address : str
        The address of the token to filter for
    token_symbol : str
        Symbol of the token (e.g., 'USDC', 'WETH')
    decimals : int
        Number of decimals for the token

    Returns:
    --------
    pd.DataFrame: Weekly aggregated token earnings by vault
    """
    liquidated, balance_updated = fetch_liquidation_data(chain)
    liquidated["week"] = pd.to_datetime(liquidated["date"]).dt.to_period("W").apply(lambda r: r.start_time)

    weth_df = liquidated[liquidated["toToken"] == WETH(chain)].copy()
    weth_df["WETH_earned"] = weth_df["amount"].astype(float) / (10**18)

    weth_weekly = weth_df.pivot_table(
        index="week", columns="vault_name", values=f"WETH_earned", aggfunc="sum", fill_value=0
    )
    weth_plot = px.bar(weth_weekly, title=f"Weekly WETH Liquidated by all autopools on chain {chain.name}")

    usdc_df = liquidated[liquidated["toToken"] == USDC(chain)].copy()
    usdc_df["USDC_earned"] = usdc_df["amount"].astype(float) / (10**6)

    usdc_weekly = usdc_df.pivot_table(
        index="week", columns="vault_name", values=f"USDC_earned", aggfunc="sum", fill_value=0
    )
    usdc_plot = px.bar(usdc_weekly, title=f"Weekly USDC Liquidated by all autopools on chain {chain.name}")
    return weth_plot, usdc_plot, liquidated, balance_updated


weth_plot, usdc_plot, liquidated, balance_updated = fetch_and_process_liquidation_by_week(ETH_CHAIN)
# for chain in ALL_CHAINS:
#     weth_plot, usdc_plot, liquidated, balance_updated = fetch_and_process_liquidation_by_week(chain)
#     weth_plot.show()
#     usdc_plot.show()

fetched logs from https://eth-mainnet 20,722,908 to 24,578,932 block_range 3,856,024 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-03-03T11:12:51.453145
fetched logs from https://eth-mainnet 20,722,908 to 24,578,932 block_range 3,856,024 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-03-03T11:12:51.963970
fetched logs from https://eth-mainnet 20,722,908 to 22,650,920 block_range 1,928,012 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-03-03T11:12:52.417228
fetched logs from https://eth-mainnet 20,722,908 to 21,686,914 block_range 964,006 status: AchemyRequestStatus.SUCCESS num logs: 4404 2026-03-03T11:12:52.758262
fetched logs from https://eth-mainnet 21,686,915 to 22,650,920 block_range 964,005 status: AchemyRequestStatus.SUCCESS num logs: 8588 2026-03-03T11:12:53.197257
fetched logs from https://eth-mainnet 22,650,921 to 24,578,932 block_range 1,928,011 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-03-03T

/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest

fetched logs from https://eth-mainnet 20,722,908 to 24,578,932 block_range 3,856,024 status: AchemyRequestStatus.SUCCESS num logs: 7969 2026-03-03T11:12:57.078304


/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest

fetched logs from https://eth-mainnet 20,722,908 to 24,578,932 block_range 3,856,024 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-03-03T11:12:58.400376
fetched logs from https://eth-mainnet 20,722,908 to 22,650,920 block_range 1,928,012 status: AchemyRequestStatus.SUCCESS num logs: 4500 2026-03-03T11:12:58.719757
fetched logs from https://eth-mainnet 22,650,921 to 24,578,932 block_range 1,928,011 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-03-03T11:12:59.087111
fetched logs from https://eth-mainnet 22,650,921 to 23,614,926 block_range 964,005 status: AchemyRequestStatus.SUCCESS num logs: 8356 2026-03-03T11:12:59.612234
fetched logs from https://eth-mainnet 23,614,927 to 24,578,932 block_range 964,005 status: AchemyRequestStatus.SUCCESS num logs: 2024 2026-03-03T11:12:59.855281


/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest

In [8]:
fluid_usdc_dv = "0x7876F91BB22148345b3De16af9448081E9853830"

fluid_usdc_balance_updated = balance_updated[balance_updated["vault"] == fluid_usdc_dv]
fluid_usdc_balance_updated

,event,address,block,transaction_index,log_index,hash,token,vault,balance,chain_id,liquidation_row,datetime
10661,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,22397526,257,607,0x5cf6c241fe6f889ecead24ec260af49226cc62381deb...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,255485509200370720893,1,0xF570EA70106B8e109222297f9a90dA477658d481,2025-05-02 17:12:23+00:00
10741,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,22400179,72,273,0xbce2735acce79ed7e0e7501847e31ef1f2b194d97e1a...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,18932555755699813472,1,0xF570EA70106B8e109222297f9a90dA477658d481,2025-05-03 02:08:11+00:00
10860,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,22406436,137,555,0x70db3e513cca6313aaecaccc398b293786e9fc4318df...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,51242629317710614856,1,0xF570EA70106B8e109222297f9a90dA477658d481,2025-05-03 23:07:47+00:00
10979,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,22412684,125,436,0x6147401a792f674fb22115d135d952d8a0877d06cb84...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,80063488475780521134,1,0xF570EA70106B8e109222297f9a90dA477658d481,2025-05-04 20:09:47+00:00
11135,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,22421014,136,390,0x2bdb970a34b7c0273a64b1a9c74ec26842e96c0af8f6...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,83594544922597917534,1,0xF570EA70106B8e109222297f9a90dA477658d481,2025-05-06 00:09:35+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
23462,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,24532310,97,474,0x9e7ced68084dd87980151c021032461685d6413cc139...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,30520635277832266208,1,0xF570EA70106B8e109222297f9a90dA477658d481,2026-02-25 07:03:23+00:00
23481,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,24539548,330,1044,0xd9e8f4fe18afb048fb4fb4da73d3444408ea01e5c0ba...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,24561956914931129427,1,0xF570EA70106B8e109222297f9a90dA477658d481,2026-02-26 07:18:35+00:00
23529,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,24560976,260,513,0xd5e911f5a3b7f73760822416def2090afbdbb054e381...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,49675844434329024688,1,0xF570EA70106B8e109222297f9a90dA477658d481,2026-03-01 07:03:47+00:00
23549,BalanceUpdated,0xF570EA70106B8e109222297f9a90dA477658d481,24568138,314,940,0x9683dd0d58a09a50e1553f6cf3eea0e19ce754eb0451...,0x6f40d4A6237C257fff2dB00FA0510DeEECd303eb,0x7876F91BB22148345b3De16af9448081E9853830,18271468095468128719,1,0xF570EA70106B8e109222297f9a90dA477658d481,2026-03-02 07:02:11+00:00


In [ ]:
break

# Couple ways to think about incentive APR.

Given at time T, we think the incentive APR of a destination, (in) is 5%. what was the realized incentive APR?

### Claimed option

what do we price the incentive otkens at?

The realized price?

the price when it was moved to the liqudation row?

if we try to do the realized price, is it difficult to seperate out incentives earned from one destiantion vs another. (for the same incentive token)




In [5]:
one_destination = "0xab3DA8995D5FeA17913c3D12A5B199F1cCC9Bf0b"
# Tokemak-USD Coin-Steakhouse High Yield USDC

In [6]:
incentive_apy = states_df[states_df["destination_vault_address"] == one_destination][["incentive_apr"]] * 100
px.scatter(
    incentive_apy, y="incentive_apr", title="Incentive APR over time for Tokemak-USD Coin-Steakhouse High Yield USDC"
)

In [7]:
from mainnet_launch.abis import ERC_20_ABI
from mainnet_launch.data_fetching.alchemy.get_events import fetch_events
import pandas as pd
from mainnet_launch.database.schema.ensure_tables_are_current.using_onchain.helpers.update_blocks import (
    ensure_all_blocks_are_in_table,
)


def get_token_transfers_to_destination(chain: ChainData, balance_updated_df):
    """
    Fetch all token transfer events to a destination vault.

    Parameters:
    -----------
    balance_updated_df : pd.DataFrame
        DataFrame containing balance updates with token_address column

    Returns:
    --------
    dict: Dictionary with token addresses as keys and transfer DataFrames as values
    """
    # can use vault liquidated and balance updated
    transfers_to_vaults = []
    for destination_vault_address in balance_updated_df["destination_vault_address"].unique():
        # Filter balance updates for the specific destination
        limited_balance_updated_df = balance_updated_df[
            balance_updated_df["destination_vault_address"] == destination_vault_address
        ]

        # Get unique token addresses
        unique_tokens = limited_balance_updated_df["token_address"].unique()
        token_contract = chain.client.eth.contract(address=unique_tokens[0], abi=ERC_20_ABI)

        transfers_to_vault = fetch_events(
            token_contract.events.Transfer,
            chain,
            argument_filters={"to": destination_vault_address},
            addresses=unique_tokens.tolist(),
        )

        ensure_all_blocks_are_in_table(transfers_to_vault["block"].to_list(), chain)
        blocks = get_full_table_as_df(Blocks, where_clause=Blocks.chain_id == chain.chain_id)

        transfers_to_vaults.append(transfers_to_vault)

    df = pd.concat(transfers_to_vaults, ignore_index=True)

    df["datetime"] = df["block"].map(blocks.set_index("block")["datetime"].to_dict())
    df["token_symbol"] = df["address"].map(tokens.set_index("token_address")["symbol"].to_dict())
    df["destination_name"] = df["to"].map(destinations.set_index("destination_vault_address")["name"].to_dict())
    df["decimals"] = df["address"].map(tokens.set_index("token_address")["decimals"].to_dict())
    df["norm_amount"] = df["value"].astype(float) / (10 ** df["decimals"])
    df = df.drop_duplicates()

    return df


transfers_to_vault = get_token_transfers_to_destination(ARBITRUM_CHAIN, balance_updated_df)
transfers_to_vault

fetched logs from https://arb-mainnet 377,406,050 to 436,964,073 block_range 59,558,023 status: AchemyRequestStatus.SUCCESS num logs: 1190 2026-02-28T11:46:24.827245
blocks_fetched
                               block  chain_id                  datetime
timestamp                                                               
2026-02-28 08:25:37+00:00  436800213     42161 2026-02-28 08:25:37+00:00
2026-02-28 14:25:38+00:00  436886895     42161 2026-02-28 14:25:38+00:00
fetched logs from https://arb-mainnet 377,406,050 to 436,964,073 block_range 59,558,023 status: AchemyRequestStatus.SUCCESS num logs: 216 2026-02-28T11:46:35.459017
fetched logs from https://arb-mainnet 377,406,050 to 436,964,073 block_range 59,558,023 status: AchemyRequestStatus.SUCCESS num logs: 260 2026-02-28T11:46:44.165060
fetched logs from https://arb-mainnet 377,406,050 to 436,964,073 block_range 59,558,023 status: AchemyRequestStatus.SUCCESS num logs: 123 2026-02-28T11:46:53.267636
fetched logs from https://arb-ma

,event,address,block,transaction_index,log_index,hash,from,to,value,datetime,token_symbol,destination_name,decimals,norm_amount
0,Transfer,0x7dfF72693f6A4149b17e7C6314655f6A9F7c8B33,378124664,2,55,0x473ef55b3206eb061027822142aa6015a99aaeaac306...,0xbA1333333333a1BA1108E8412f11850A5C319bA9,0xce1c8244410a4F97308Ffc5Ec926C9EF8FAeC809,47517690565670610418,2025-09-11 19:32:08+00:00,GHO,Tokemak-USD Coin-Balancer Aave GHO/USDT/USDC,18,47.517691
1,Transfer,0x040d1EdC9569d4Bab2D15287Dc5A4F10F56a56B8,378159085,3,1,0x31e4641edd54611c0d3e51867d0838ffe4cf9ccfa3bc...,0x627305fF273640e0468D3c5DFE4FC2Adb1D013e8,0xce1c8244410a4F97308Ffc5Ec926C9EF8FAeC809,11896196644011366,2025-09-11 21:55:57+00:00,BAL,Tokemak-USD Coin-Balancer Aave GHO/USDT/USDC,18,0.011896
2,Transfer,0x1509706a6c66CA549ff0cB464de88231DDBe213B,378159085,3,2,0x31e4641edd54611c0d3e51867d0838ffe4cf9ccfa3bc...,0xeC1c780A275438916E7CEb174D80878f29580606,0xce1c8244410a4F97308Ffc5Ec926C9EF8FAeC809,11177460364823330,2025-09-11 21:55:57+00:00,AURA,Tokemak-USD Coin-Balancer Aave GHO/USDT/USDC,18,0.011177
3,Transfer,0x1509706a6c66CA549ff0cB464de88231DDBe213B,378159085,3,4,0x31e4641edd54611c0d3e51867d0838ffe4cf9ccfa3bc...,0x3C25C7D4B430a9eEcfca664B2B396C9DC9b0f865,0xce1c8244410a4F97308Ffc5Ec926C9EF8FAeC809,10452457262176640,2025-09-11 21:55:57+00:00,AURA,Tokemak-USD Coin-Balancer Aave GHO/USDT/USDC,18,0.010452
4,Transfer,0x7dfF72693f6A4149b17e7C6314655f6A9F7c8B33,378159085,3,6,0x31e4641edd54611c0d3e51867d0838ffe4cf9ccfa3bc...,0x49adA06908039FdC22cbd4fdFF9010A934e58382,0xce1c8244410a4F97308Ffc5Ec926C9EF8FAeC809,99196682968205667,2025-09-11 21:55:57+00:00,GHO,Tokemak-USD Coin-Balancer Aave GHO/USDT/USDC,18,0.099197
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1868,Transfer,0x912CE59144191C1204E64559FE8253a0e49E6548,383984030,1,0,0xdf0b1fd55c1f62789a5ddc8c3a0381a42e0ddf9f4783...,0x3Ef3D8bA38EBe18DB133cEc108f4D14CE00Dd9Ae,0xd70F872E4Fd8A55076082Ff73ED7F54524D6c74b,112919618754655410094,2025-09-28 18:05:14+00:00,ARB,Tokemak-USD Coin-Fluid Tether USD,18,112.919619
1869,Transfer,0x61E030A56D33e8260FdD81f03B162A79Fe3449Cd,384236057,4,16,0x1e9ababad5700d9b3b8be0c44edc818520787892b8a5...,0x94312a608246Cecfce6811Db84B3Ef4B2619054E,0xd70F872E4Fd8A55076082Ff73ED7F54524D6c74b,4069619677511506828,2025-09-29 11:34:22+00:00,FLUID,Tokemak-USD Coin-Fluid Tether USD,18,4.069620
1870,Transfer,0x912CE59144191C1204E64559FE8253a0e49E6548,384329986,3,6,0xfb7653cb5119554f0ccc0f84486d5272de213bdb9f5f...,0x3Ef3D8bA38EBe18DB133cEc108f4D14CE00Dd9Ae,0xd70F872E4Fd8A55076082Ff73ED7F54524D6c74b,56028704729802639967,2025-09-29 18:05:03+00:00,ARB,Tokemak-USD Coin-Fluid Tether USD,18,56.028705
1871,Transfer,0x61E030A56D33e8260FdD81f03B162A79Fe3449Cd,384582106,1,4,0xaa5d0ceac2dee496713f31a1a933c11a9a29c858c1b1...,0x94312a608246Cecfce6811Db84B3Ef4B2619054E,0xd70F872E4Fd8A55076082Ff73ED7F54524D6c74b,3917819417112638355,2025-09-30 11:34:22+00:00,FLUID,Tokemak-USD Coin-Fluid Tether USD,18,3.917819


fetched logs from https://eth-mainnet 20,722,908 to 24,557,703 block_range 3,834,795 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:03.469568
fetched logs from https://eth-mainnet 20,722,908 to 24,557,703 block_range 3,834,795 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-02-28T12:06:03.865504
fetched logs from https://eth-mainnet 20,722,908 to 22,640,305 block_range 1,917,397 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-02-28T12:06:04.244004
fetched logs from https://eth-mainnet 20,722,908 to 21,681,606 block_range 958,698 status: AchemyRequestStatus.SUCCESS num logs: 4385 2026-02-28T12:06:04.617418
fetched logs from https://eth-mainnet 21,681,607 to 22,640,305 block_range 958,698 status: AchemyRequestStatus.SUCCESS num logs: 8527 2026-02-28T12:06:05.096360
fetched logs from https://eth-mainnet 22,640,306 to 24,557,703 block_range 1,917,397 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-02-28T

/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest

fetched logs from https://eth-mainnet 20,722,908 to 24,557,703 block_range 3,834,795 status: AchemyRequestStatus.SUCCESS num logs: 7969 2026-02-28T12:06:08.819061


/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest

fetched logs from https://eth-mainnet 20,722,908 to 24,557,703 block_range 3,834,795 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-02-28T12:06:10.169089
fetched logs from https://eth-mainnet 20,722,908 to 22,640,305 block_range 1,917,397 status: AchemyRequestStatus.SUCCESS num logs: 4421 2026-02-28T12:06:11.066481
fetched logs from https://eth-mainnet 22,640,306 to 24,557,703 block_range 1,917,397 status: AchemyRequestStatus.SPLIT_RANGE_AND_TRY_AGAIN num logs: 0 2026-02-28T12:06:11.412320
fetched logs from https://eth-mainnet 22,640,306 to 23,599,004 block_range 958,698 status: AchemyRequestStatus.SUCCESS num logs: 8365 2026-02-28T12:06:11.849205
fetched logs from https://eth-mainnet 23,599,005 to 24,557,703 block_range 958,698 status: AchemyRequestStatus.SUCCESS num logs: 2027 2026-02-28T12:06:12.115061


...

fetched logs from https://sonic-mainnet 45,593,624 to 47,593,623 block_range 1,999,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:31.628544fetched logs from https://sonic-mainnet 35,593,624 to 37,593,623 block_range 1,999,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:31.630295

fetched logs from https://sonic-mainnet 37,593,624 to 39,593,623 block_range 1,999,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:31.632569
fetched logs from https://sonic-mainnet 31,593,624 to 33,593,623 block_range 1,999,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:31.633359
fetched logs from https://sonic-mainnet 41,593,624 to 43,593,623 block_range 1,999,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:31.635257
fetched logs from https://sonic-mainnet 49,593,624 to 51,593,623 block_range 1,999,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:31.637634
fetched logs from https://so

fetched logs from https://arb-mainnet 377,406,050 to 436,968,377 block_range 59,562,327 status: AchemyRequestStatus.SUCCESS num logs: 1531 2026-02-28T12:06:37.079679
fetched logs from https://arb-mainnet 377,406,050 to 436,968,377 block_range 59,562,327 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:37.655998
fetched logs from https://arb-mainnet 377,406,050 to 436,968,377 block_range 59,562,327 status: AchemyRequestStatus.SUCCESS num logs: 1431 2026-02-28T12:06:37.901808
fetched logs from https://arb-mainnet 377,406,050 to 436,968,377 block_range 59,562,327 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:38.349568


fetched logs from https://plasma-mainnet 1,385,809 to 1,395,808 block_range 9,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:40.533649
fetched logs from https://plasma-mainnet 1,425,809 to 1,435,808 block_range 9,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:40.540866
fetched logs from https://plasma-mainnet 1,445,809 to 1,455,808 block_range 9,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:40.542205
fetched logs from https://plasma-mainnet 1,455,809 to 1,465,808 block_range 9,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:40.543816
fetched logs from https://plasma-mainnet 1,395,809 to 1,405,808 block_range 9,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:40.545665
fetched logs from https://plasma-mainnet 1,465,809 to 1,475,808 block_range 9,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:06:40.546549
fetched logs from https://plasma-mainnet 1,435,809 to 1,44

fetched logs from https://linea-mainnet 26,833,829 to 27,083,828 block_range 249,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:08:29.596229
fetched logs from https://linea-mainnet 27,083,829 to 27,333,828 block_range 249,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:08:29.609372
fetched logs from https://linea-mainnet 26,583,829 to 26,833,828 block_range 249,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:08:29.613295
fetched logs from https://linea-mainnet 27,333,829 to 27,583,828 block_range 249,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:08:29.737221
fetched logs from https://linea-mainnet 27,583,829 to 27,833,828 block_range 249,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:08:29.747559
fetched logs from https://linea-mainnet 27,833,829 to 28,083,828 block_range 249,999 status: AchemyRequestStatus.SUCCESS num logs: 0 2026-02-28T12:08:29.749949
fetched logs from https://linea-mainnet 

KeyError: 'block'

In [9]:
base_destinations = destinations[
    (destinations["chain_id"] == BASE_CHAIN.chain_id) & (destinations["denominated_in"] == USDC(BASE_CHAIN))
]
base_destinations

,destination_vault_address,chain_id,exchange_name,name,symbol,pool_type,pool,underlying,underlying_symbol,underlying_name,denominated_in,destination_vault_decimals,block_deployed
161,0xb5F780EC144A0f06ca1820E14A5611EA7E464491,8453,morpho,Tokemak-USD Coin-Spark USDC Vault,toke-USDC-sparkUSDC,metaMorpho,0x7BfA7C4f149E7415b73bdeDfe609237e29CBF34A,0x7BfA7C4f149E7415b73bdeDfe609237e29CBF34A,sparkUSDC,Spark USDC Vault,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,36628392
162,0x30Ff0dBBd64A86C8F5Bfe905A53f0B78cBb6Fe5b,8453,curve,Tokemak-USD Coin-USDC/Savings crvUSD,toke-USDC-scrvUSDC,curveNG,0x5aB01ee6208596f2204B85bDFA39d34c2aDD98F6,0x5aB01ee6208596f2204B85bDFA39d34c2aDD98F6,scrvUSDC,USDC/Savings crvUSD,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,30313679
163,0xeE23855e36713579FB0af48120557dF855280B60,8453,euler,Tokemak-USD Coin-EVK Vault eUSDC-1,toke-USDC-eUSDC-1,eVault,0x0A1a3b5f2041F33522C4efc754a7D096f880eE16,0x0A1a3b5f2041F33522C4efc754a7D096f880eE16,eUSDC-1,EVK Vault eUSDC-1,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,6,30565692
164,0xFd70A9b4cE03Dc07Fe7Ba5e4D39fac11eE8d8615,8453,morpho,Tokemak-USD Coin-Moonwell Flagship USDC,toke-USDC-mwUSDC,metaMorpho,0xc1256Ae5FF1cf2719D4937adb3bbCCab2E00A2Ca,0xc1256Ae5FF1cf2719D4937adb3bbCCab2E00A2Ca,mwUSDC,Moonwell Flagship USDC,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,30313679
168,0xF37860D12eab28716C6f8d1Eb92D599Af38C8cE2,8453,balancerV3,Tokemak-USD Coin-Balancer Aave USDC-Aave GHO,toke-USDC-Aave USDC-Aave GHO,balV3,0x7AB124EC4029316c2A42F713828ddf2a192B36db,0x7AB124EC4029316c2A42F713828ddf2a192B36db,Aave USDC-Aave GHO,Balancer Aave USDC-Aave GHO,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,30565692
171,0xd3a721711b41c4f22C22A0727853fE322B84e931,8453,morpho,Tokemak-USD Coin-Steakhouse USDC,toke-USDC-steakUSDC,metaMorpho,0xbeeF010f9cb27031ad51e3333f9aF9C6B1228183,0xbeeF010f9cb27031ad51e3333f9aF9C6B1228183,steakUSDC,Steakhouse USDC,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,30313679
175,0x4eeDC6d6126ae9C6D92D03bB355eEAe5E47d2b03,8453,morpho,Tokemak-USD Coin-Gauntlet USDC Prime,toke-USDC-gtUSDCp,metaMorpho,0xeE8F4eC5672F09119b96Ab6fB59C27E1b7e44b61,0xeE8F4eC5672F09119b96Ab6fB59C27E1b7e44b61,gtUSDCp,Gauntlet USDC Prime,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,30313679
176,0x5d2B4Fc5f8975d9EC10024b04Dc71E6ba61707DC,8453,aave,Tokemak-USD Coin-Wrapped Aave Base USDC,toke-USDC-waBasUSDC,aTokenStataV3,0xC768c589647798a6EE01A91FdE98EF2ed046DBD6,0xC768c589647798a6EE01A91FdE98EF2ed046DBD6,waBasUSDC,Wrapped Aave Base USDC,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,6,30565692
178,0x852f5aB62b0b6526980450272C1dB02b48BF7a60,8453,balancerV3,Tokemak-USD Coin-Balancer Aave GHO-USR,toke-USDC-Aave GHO-USR,balV3,0x5b14ce8DE84448e9A6F6aF652af318472aA4FCAF,0x5b14ce8DE84448e9A6F6aF652af318472aA4FCAF,Aave GHO-USR,Balancer Aave GHO-USR,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,30565692
179,0x713bB72e9405F46e047185D9634C7fCDbDC10eED,8453,balancerV3,Tokemak-USD Coin-Balancer Aave USDC-Aave GHO,toke-USDC-Aave USDC-Aave GHO,balV3,0x7AB124EC4029316c2A42F713828ddf2a192B36db,0x7AB124EC4029316c2A42F713828ddf2a192B36db,Aave USDC-Aave GHO,Balancer Aave USDC-Aave GHO,0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913,18,30313679


In [10]:
import pandas as pd
from pathlib import Path
from multicall import Call

from mainnet_launch.data_fetching.get_state_by_block import (
    get_state_by_one_block,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    get_raw_state_by_blocks,
    safe_normalize_6_with_bool_success,
)
from mainnet_launch.constants import ETH_CHAIN, BASE_CHAIN


blocks = build_blocks_to_use(start_block=37410118, chain=BASE_CHAIN)

import numpy as np

# Generate 24 evenly spaced blocks between each consecutive pair of blocks
expanded_blocks = []
for i in range(len(blocks) - 1):
    # Get current and next block
    current_block = blocks[i]
    next_block = blocks[i + 1]

    # Generate 24 intermediate blocks (plus the current block = 25 points total between pairs)
    intermediate_blocks = np.linspace(current_block, next_block, 26, dtype=int)[:-1]
    expanded_blocks.extend(intermediate_blocks)

# Add the last block
expanded_blocks.append(blocks[-1])

# Tokemak-USD Coin-Steakhouse High Yield Instant
steakhouse = "0xbeeff7aE5E00Aae3Db302e4B0d8C883810a58100"
surge_call = Call(
    steakhouse,
    ["convertToAssets(uint256)(uint256)", int(1e18)],
    [("steakhouse_high_yield_instant", safe_normalize_6_with_bool_success)],
)

df = get_raw_state_by_blocks([surge_call], expanded_blocks, BASE_CHAIN, include_block_number=True)
df

px.scatter(df[df.index.to_series().dt.date == pd.to_datetime("2026-02-21").date()])

Cannot connect to host base-mainnet.g.alchemy.com:443 ssl:default [nodename nor servname provided, or not known] [0]
Cannot connect to host base-mainnet.g.alchemy.com:443 ssl:default [nodename nor servname provided, or not known] [0]
Cannot connect to host base-mainnet.g.alchemy.com:443 ssl:default [nodename nor servname provided, or not known] [0]


In [11]:
import plotly.express as px

# Calculate actual APY looking forward (realized returns)
df["actual_apy_1d"] = (
    ((df["steakhouse_high_yield_instant"].shift(-1) / df["steakhouse_high_yield_instant"]) - 1) * (365 / 1) * 100
)
df["actual_apy_5d"] = (
    ((df["steakhouse_high_yield_instant"].shift(-5) / df["steakhouse_high_yield_instant"]) - 1) * (365 / 5) * 100
)
df["actual_apy_30d"] = (
    ((df["steakhouse_high_yield_instant"].shift(-30) / df["steakhouse_high_yield_instant"]) - 1) * (365 / 30) * 100
)

# Calculate backward-looking APY (expected based on past performance)
df["pred_apy_1d"] = (
    ((df["steakhouse_high_yield_instant"] / df["steakhouse_high_yield_instant"].shift(1)) - 1) * (365 / 1) * 100
)
df["pred_apy_5d"] = (
    ((df["steakhouse_high_yield_instant"] / df["steakhouse_high_yield_instant"].shift(5)) - 1) * (365 / 5) * 100
)
df["pred_apy_30d"] = (
    ((df["steakhouse_high_yield_instant"] / df["steakhouse_high_yield_instant"].shift(30)) - 1) * (365 / 30) * 100
)

# Plot the APYs that exist in the dataframe
# Filter dataframe to only include data after Oct 25, 2025
df = df[df.index > "2025-10-25"]
fig = px.line(
    df,
    y=["actual_apy_1d", "actual_apy_5d", "actual_apy_30d", "pred_apy_1d", "pred_apy_5d", "pred_apy_30d"],
    title="Steakhouse High Yield Instant APY (%) - Multiple Time Periods",
    labels={"value": "APY (%)", "variable": "Time Period"},
)
fig.show()

In [12]:
# Calculate prediction errors
df["error_1d"] = df["actual_apy_1d"] - df["pred_apy_1d"]
df["error_5d"] = df["actual_apy_5d"] - df["pred_apy_5d"]
df["error_30d"] = df["actual_apy_30d"] - df["pred_apy_30d"]

# Create separate histograms for the prediction errors
# Create ECDF plots for prediction errors
import plotly.graph_objects as go
import numpy as np


# Create separate histograms for each error column
fig1 = px.histogram(
    df,
    x="error_1d",
    title="Distribution of 1 Day APY Prediction Errors",
    labels={"error_1d": "Prediction Error (Actual APY - Predicted APY) %"},
)
fig1.show()

fig2 = px.histogram(
    df,
    x="error_5d",
    title="Distribution of 5 Day APY Prediction Errors",
    labels={"error_5d": "Prediction Error (Actual APY - Predicted APY) %"},
)
fig2.show()

fig3 = px.histogram(
    df,
    x="error_30d",
    title="Distribution of 30 Day APY Prediction Errors",
    labels={"error_30d": "Prediction Error (Actual APY - Predicted APY) %"},
)
fig3.show()

In [13]:
# Get descriptive statistics for all error columns
error_stats = df[["error_1d", "error_5d", "error_30d"]].describe()
error_stats.round(2)

,error_1d,error_5d,error_30d
count,3074.00,3066.00,3016.00
mean,0.00,0.00,-0.00
std,0.22,0.09,0.01
min,-3.65,-0.73,-0.12
25%,-0.00,-0.01,-0.00
50%,-0.00,-0.00,-0.00
75%,0.00,0.01,0.00
max,3.65,0.85,0.12
